In [1]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az

import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('data/scr_brain_group.csv')

In [5]:
# %% Detection-rate-adjusted model (amygdala–hippocampus)
# Adds standardized per-subject zero-SCR count as a covariate to the
# group × coupling interaction model, to address R2 #5 (SCR detection confound).

df['sub_idx'] = pd.Categorical(df['sub']).codes
n_subs = df['sub_idx'].nunique()

group_order = ['HC', 'VCC', 'VPTSD']
df['group'] = pd.Categorical(df['group'], categories=group_order, ordered=True)
df['group_idx'] = df['group'].cat.codes
n_groups = df['group_idx'].nunique()

# Standardize the detection covariate (subject-constant, but stored per row)
df['n_zero_z'] = (df['n_zero_scr_x'] - df['n_zero_scr_x'].mean()) / df['n_zero_scr_x'].std()

pe        = df['pe'].values
coupling  = df['amg_vmpfc'].values
amg       = df['amg'].values
trialNo   = df['trialNo'].values
sub_idx   = df['sub_idx'].values
group_idx = df['group_idx'].values
n_zero_z  = df['n_zero_z'].values

with pm.Model() as model_adj:
    beta_coupling = pm.Normal('beta_coupling', 0, 1)
    beta_amg      = pm.Normal('beta_amg', 0, 1)
    beta_trialNo  = pm.Normal('beta_trialNo', 0, 1)
    beta_nzero    = pm.Normal('beta_nzero', 0, 1)          # detection-rate covariate

    beta_group_raw = pm.Normal('beta_group_raw', 0, 1, shape=n_groups - 1)
    beta_group = pm.math.concatenate([[0], beta_group_raw])

    beta_interaction_raw = pm.Normal('beta_interaction_raw', 0, 1, shape=n_groups - 1)
    beta_interaction = pm.math.concatenate([[0], beta_interaction_raw])

    mu_a = pm.Normal('mu_a', 0, 1)
    sigma_a = pm.HalfNormal('sigma_a', 1)
    z_a = pm.Normal('z_a', 0, 1, shape=n_subs)
    a = pm.Deterministic('a', mu_a + z_a * sigma_a)

    mu = (
        a[sub_idx] +
        beta_group[group_idx] +
        beta_coupling * coupling +
        beta_interaction[group_idx] * coupling +
        beta_amg * amg +
        beta_trialNo * trialNo +
        beta_nzero * n_zero_z                              # <-- adjustment
    )

    sigma = pm.HalfNormal('sigma', 1)
    pm.Normal('pe', mu=mu, sigma=sigma, observed=pe)
    trace_adj = pm.sample(chains=4, random_seed=42,
                          return_inferencedata=True,
                          idata_kwargs={"log_likelihood": True})

az.summary(trace_adj,
           var_names=['beta_coupling', 'beta_group_raw',
                      'beta_interaction_raw', 'beta_nzero'],
           hdi_prob=0.89)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_coupling, beta_amg, beta_trialNo, beta_nzero, beta_group_raw, beta_interaction_raw, mu_a, sigma_a, z_a, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 10 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


,mean,sd,hdi_5.5%,hdi_94.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta_coupling,-0.044,0.033,-0.097,0.008,0.001,0.000,2642.0,2856.0,1.0
beta_group_raw[0],0.011,0.023,-0.025,0.048,0.000,0.000,2764.0,3003.0,1.0
beta_group_raw[1],-0.032,0.023,-0.071,0.003,0.000,0.000,3251.0,2714.0,1.0
beta_interaction_raw[0],-0.037,0.044,-0.105,0.034,0.001,0.001,2666.0,3026.0,1.0
beta_interaction_raw[1],0.081,0.045,0.016,0.159,0.001,0.001,3115.0,3059.0,1.0
beta_nzero,-0.003,0.007,-0.014,0.008,0.000,0.000,10561.0,2817.0,1.0


In [6]:
# %% Adjusted group-specific slopes + the headline contrast
post = trace_adj.posterior
slope_HC    = post['beta_coupling']
slope_VCC   = post['beta_coupling'] + post['beta_interaction_raw'][:, :, group_order.index('VCC') - 1]
slope_VPTSD = post['beta_coupling'] + post['beta_interaction_raw'][:, :, group_order.index('VPTSD') - 1]

diff = slope_VCC - slope_VPTSD
pd_diff = float((diff < 0).mean())
print(f"ADJUSTED VCC-VPTSD: mean={float(diff.mean()):.3f}, sd={float(diff.std()):.3f}, "
      f"pd={pd_diff*100:.1f}%, 89% HDI={az.hdi(diff.values.flatten(), hdi_prob=0.89)}")

ADJUSTED VCC-VPTSD: mean=-0.118, sd=0.041, pd=99.8%, 89% HDI=[-0.18629161 -0.05604927]
